# TEKNOFEST KPI — Google Colab (GPU)

Mac yorulmasın diye ağır iş burada: **18 videoluk KPI + Qwen-VL**.

## Senin Mac’te yapacağın (bir kez)

1. Google Drive’da klasör aç: `KAIZEN_KPI/data/`
2. Şunları yükle (Finder → Drive web):
   - `data/videos/` (accident / near_miss / normal klasörleriyle)
   - `data/exports/gold_labels_hepsi.json`
3. Bu defteri Colab’e yükle **veya** repoyu klonlayınca `notebooks/` içinden aç.

## Colab’de

**Runtime → Change runtime type → T4 GPU** seç, sonra hücreleri sırayla çalıştır.

### 1) GPU kontrolü

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'T4 GPU seçili değil — Runtime ayarından GPU aç.'

### 2) Google Drive bağla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/KAIZEN_KPI'
!mkdir -p "{DRIVE_ROOT}/data/videos" "{DRIVE_ROOT}/data/exports" "{DRIVE_ROOT}/data/predictions_wide"
print('Drive kökü:', DRIVE_ROOT)

### 3) Repoyu çek

`zehra/calisma` dalı henüz GitHub’da yoksa `main` kullan; KPI scriptleri push edilmiş olmalı.
Push etmediysen: Mac’ten zip atıp Drive’a koy, aşağıdaki **alternatif** hücreyi kullan.

In [ ]:
REPO = '/content/teknofest-video-ajan'
REPO_URL = 'https://github.com/TulinBabalikKopmaz/KAIZEN_Teknofest26_DilAjanlar-_VideoAnalizKararSistemi.git'
BRANCH = 'main'  # KPI kodları push edilince: zehra/calisma

import os
if not os.path.exists(REPO):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO}
else:
    %cd {REPO}
    !git fetch --depth 1 origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
%cd {REPO}
!ls scripts/run_kpi_wide.py scripts/colab_kpi_bootstrap.py

#### Alternatif: Drive’daki repo zip’i (GitHub güncel değilse)

Mac’te projeyi zipleyip `MyDrive/KAIZEN_KPI/repo.zip` olarak yükle, sonra bu hücreyi çalıştır.

In [ ]:
# Sadece zip kullanacaksan bu hücreyi çalıştır (yukarıdaki clone yerine):
# ZIP = '/content/drive/MyDrive/KAIZEN_KPI/repo.zip'
# REPO = '/content/teknofest-video-ajan'
# !rm -rf {REPO} && mkdir -p {REPO} && unzip -q {ZIP} -d /content/_unzipped
# # zip içinde tek klasör varsa onu REPO'ya taşı
# import os, shutil, glob
# roots = [p for p in glob.glob('/content/_unzipped/*') if os.path.isdir(p)]
# src = roots[0] if len(roots)==1 else '/content/_unzipped'
# !rm -rf {REPO}
# shutil.move(src, REPO)
# %cd {REPO}
print('Zip alternatifi şimdilik kapalı — gerekirse yorum satırlarını aç.')

### 4) Ortamı kur (Ollama + Qwen + YOLO + Drive bağlantısı)

İlk seferde model indirme ~5–10 dk sürebilir.

In [ ]:
%cd /content/teknofest-video-ajan
!python scripts/colab_kpi_bootstrap.py \
  --drive-root /content/drive/MyDrive/KAIZEN_KPI \
  --repo /content/teknofest-video-ajan \
  --model qwen2.5vl:7b

### 5) Drive’da veri var mı kontrol et

In [ ]:
from pathlib import Path
root = Path('/content/teknofest-video-ajan/data')
vids = list(root.joinpath('videos').rglob('*.mp4'))
gold = root / 'exports' / 'gold_labels_hepsi.json'
print('video:', len(vids))
print('gold :', gold.exists(), gold)
assert len(vids) >= 18, 'Drive\'a en az 18 mp4 yükleyin (data/videos/...).'
assert gold.exists(), 'gold_labels_hepsi.json eksik — Mac\'ten Drive/exports\'a kopyalayın.'

### 6) Geniş KPI çalıştır (18 video)

Mac’teki gibi: 6 kaza + 6 near miss + 6 normal. Sonuçlar Drive’a yazılır.

Süre: T4’te kabaca **30–90 dk** (modele ve kare sayısına göre).

In [ ]:
%cd /content/teknofest-video-ajan
!python -u scripts/run_kpi_wide.py \
  --n 18 \
  --seed 42 \
  --model qwen2.5vl:7b \
  --pred-dir data/predictions_wide \
  --no-second-look

print('--- özet ---')
!ls -la data/exports/kpi_wide*.csv 2>/dev/null || true
!tail -n 20 data/exports/kpi_wide_qwen2_5vl_7b_ozet.csv 2>/dev/null || \
  !tail -n 20 data/exports/kpi_wide_7b_ozet.csv 2>/dev/null || \
  !ls data/exports/

### 7) (İsteğe bağlı) Daha büyük model — Colab’de

Mac’te 32B / LLaVA sorun çıkardı. Colab T4’te önce **7B** ile skor al.
Daha büyüğü denemek için (VRAM yetmezse küçült):

```bash
!ollama pull qwen2.5vl:7b   # güvenli
# !ollama pull llava:13b    # deneme; 2 kare ile
```

LLaVA için:
```bash
!python -u scripts/run_kpi_wide.py --n 18 --seed 42 --model llava:13b \
  --pred-dir data/predictions_wide_llava13b --no-second-look --max-frames 2
```

### Mac’e ne kalır?

- Streamlit ile sonuçları izlemek
- Drive’dan `kpi_wide_*_ozet.csv` indirmek
- Kod yazmak / gold düzeltmek

**Ağır model + 18 video = Colab.** Mac’i kapatmana gerek kalmaz.